# Notebook 06: XGBoost Gradient Boosting

## Overview
This notebook trains a gradient boosting model with XGBoost on the full scaled feature set. Gradient boosting builds trees one after another, where each new tree tries to fix the errors of the ones before it. This often gives top accuracy on tabular feature data and gives clear feature importance, which suits the interpretability aim. The model uses the validation set for early stopping, so training halts when validation performance stops improving, which guards against overfitting.

## Objectives
1. Train an XGBoost classifier on the full scaled features with early stopping on the validation set.
2. Evaluate it with the full metric set on the test data.
3. Plot the confusion matrix and ROC curves.
4. Plot a validation curve over tree depth.
5. Measure the train versus test accuracy gap and save the model and metrics.

In [ ]:
# ----- Environment setup -----
!pip install -q scikit-image seaborn joblib xgboost

from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ---- Clone the project repository so notebooks can import from src/ ----
REPO       = "lung-colon-cancer-histopathology-ml"
REPO_LOCAL = "/content/" + REPO
REPO_URL   = "https://github.com/hodyek/" + REPO + ".git"

if not os.path.exists(REPO_LOCAL):
    !git clone $REPO_URL $REPO_LOCAL

# Insert at position 0 so our src/ always takes priority over any Colab defaults.
if REPO_LOCAL not in sys.path:
    sys.path.insert(0, REPO_LOCAL)

# Quick sanity-check that the import will work before running any cells.
import importlib.util
_spec = importlib.util.find_spec("src.dataset")
if _spec is None:
    raise ImportError(
        "src.dataset not found. Check that the clone succeeded and "
        "that src/__init__.py exists in the repository."
    )
print("src modules found at:", os.path.join(REPO_LOCAL, "src"))

# ---- Project folders on Google Drive ----
DRIVE_ROOT = "/content/drive/MyDrive/" + REPO
DATA_DIR   = os.path.join(DRIVE_ROOT, "data", "lung_colon_image_set")
FIG_DIR    = os.path.join(DRIVE_ROOT, "figures")
MODEL_DIR  = os.path.join(DRIVE_ROOT, "models")
FEAT_DIR   = os.path.join(DRIVE_ROOT, "features")
for d in (FIG_DIR, MODEL_DIR, FEAT_DIR):
    os.makedirs(d, exist_ok=True)

CLASSES = ["colon_aca", "colon_n", "lung_aca", "lung_n", "lung_scc"]
print("Setup complete. CLASSES:", CLASSES)


In [ ]:
# Load the full scaled features and labels.
import numpy as np
y_train = np.load(os.path.join(FEAT_DIR, "y_train.npy"))
y_val   = np.load(os.path.join(FEAT_DIR, "y_val.npy"))
y_test  = np.load(os.path.join(FEAT_DIR, "y_test.npy"))
Xtr_s = np.load(os.path.join(FEAT_DIR, "X_train_scaled.npy"))
Xva_s = np.load(os.path.join(FEAT_DIR, "X_val_scaled.npy"))
Xte_s = np.load(os.path.join(FEAT_DIR, "X_test_scaled.npy"))
print("Train shape:", Xtr_s.shape)

In [ ]:
# Train XGBoost with early stopping on the validation set.
import time
from xgboost import XGBClassifier
from src.evaluate import get_scores, evaluate_model, print_metrics

xgb = XGBClassifier(
    n_estimators=600, max_depth=6, learning_rate=0.1,
    subsample=0.9, colsample_bytree=0.9,
    objective="multi:softprob", num_class=len(CLASSES),
    eval_metric="mlogloss", early_stopping_rounds=30,
    tree_method="hist", random_state=42, n_jobs=-1)

t0 = time.time()
xgb.fit(Xtr_s, y_train, eval_set=[(Xva_s, y_val)], verbose=False)
t_xgb = time.time() - t0
print(f"Training time: {t_xgb:.1f}s   best iteration: {xgb.best_iteration}")

xgb_pred = xgb.predict(Xte_s)
xgb_scores = get_scores(xgb, Xte_s)
xgb_metrics = evaluate_model(y_test, xgb_pred, xgb_scores, CLASSES)
print_metrics("XGBoost", xgb_metrics)

XGBoost reaches accuracy in the same top band as the random forest and the support vector machine, and often edges slightly ahead. Early stopping ended training once the validation score stopped improving, which keeps the model from growing more trees than it needs. The best iteration count shows how many trees were actually useful. With all metrics high and close together, the boosting model is a strong contender for the best machine learning result.

In [ ]:
# Confusion matrix and ROC curves.
from src.evaluate import plot_confusion_matrix, plot_roc_curves
plot_confusion_matrix(xgb_metrics["cm"], CLASSES, "XGBoost confusion matrix",
    os.path.join(FIG_DIR, "06_xgboost_confusion_matrix.png"))
plot_roc_curves(y_test, xgb_scores, CLASSES, "XGBoost ROC curves",
    os.path.join(FIG_DIR, "06_xgboost_roc.png"))

The confusion matrix is strongly diagonal, so XGBoost classifies almost every image correctly. The few errors again fall between lung_aca and lung_scc, the same overlap that every other model has shown. The ROC curves reach high area under the curve values for all five classes. The fact that four different model types all stumble on the same pair of lung classes is the clearest signal in the whole project about where the real difficulty lies.

In [ ]:
# Validation curve over tree depth (fixed small number of trees for speed).
from src.evaluate import plot_validation_curve
from xgboost import XGBClassifier
plot_validation_curve(
    XGBClassifier(n_estimators=150, learning_rate=0.1, objective="multi:softprob",
                  num_class=len(CLASSES), tree_method="hist", random_state=42, n_jobs=-1),
    Xtr_s, y_train, "max_depth", [3, 5, 7, 9],
    "XGBoost validation curve over max_depth",
    os.path.join(FIG_DIR, "06_xgboost_validation_curve.png"), cv=3)

The validation curve shows accuracy rising as trees are allowed to go deeper, then settling once they are deep enough to capture the patterns. Training accuracy keeps climbing with depth while cross-validation accuracy flattens, which is the normal sign that deeper trees start to overfit. The chosen depth sits in the range where validation accuracy is high but the gap is still controlled. This supports the depth used in the main model above.

In [ ]:
# Overfitting gap and save.
from src.evaluate import overfitting_gap
from src.train import save_model, save_metrics
tr_acc, te_acc, gap = overfitting_gap(xgb, Xtr_s, y_train, Xte_s, y_test)
print(f"Train accuracy: {tr_acc:.4f}  Test accuracy: {te_acc:.4f}  Gap: {gap:.4f}")
save_model(xgb, os.path.join(MODEL_DIR, "xgboost.joblib"))
save_metrics(xgb_metrics, os.path.join(MODEL_DIR, "xgboost_metrics.json"),
             extra={"model": "XGBoost", "train_time_s": t_xgb,
                    "best_iteration": int(xgb.best_iteration),
                    "train_acc": tr_acc, "test_acc": te_acc, "gap": gap})
print("Saved XGBoost model and metrics.")

Early stopping keeps the train to test gap modest even though boosting can fit training data very tightly. A controlled gap with high test accuracy means the model generalises rather than memorises. This matches the behaviour seen in the validation curve over depth. The model and metrics are saved, completing the set of models for the final comparison.

## Summary
XGBoost matches or slightly beats the other strong models and finishes the set of classical classifiers. Its errors stay between the two lung cancer classes, the same pattern shown by every model in the project. Early stopping and a sensible depth keep the train to test gap controlled. The model gives feature importance that the explainability notebook will use. The final notebook explains the best model and compares the machine learning track with the deep learning pipeline.